In [2]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 6.0 MB/s eta 0:00:00


In [62]:
import os
import openai 

## Getting my API key
openai.api_key = os.getenv("sk-proj-oycWS74JHu9O7z_JUIvcInJkpxcwJ")

def ask_chatbot(prompt):
    response = openai.ChatCompletion.create(
        model = "gpt-3.5-turbo",
        messages = [
            {"role": "system", "content": "You are helpful, witty librarian from the future who manages a vault of time-travel artifacts."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response['choices'][0]['message']['content']

In [ ]:
import sqlite3

# -------------------------
# DATABASE MANAGER
# -------------------------
class DatabaseManager:
    def __init__(self, db_name="vault.db"):
        self.conn = sqlite3.connect(db_name)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                user_id INTEGER PRIMARY KEY,
                name TEXT NOT NULL
            )
        ''')
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS items (
                item_id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL UNIQUE,
                origin_era TEXT NOT NULL,
                max_duration INTEGER NOT NULL,
                is_borrowed INTEGER DEFAULT 0
            )
        ''')
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS borrowed_items (
                user_id INTEGER,
                item_id INTEGER,
                turns_borrowed INTEGER DEFAULT 0,
                PRIMARY KEY (user_id, item_id),
                FOREIGN KEY (user_id) REFERENCES users(user_id),
                FOREIGN KEY (item_id) REFERENCES items(item_id)
            )
        ''')
        self.conn.commit()

    def add_user(self, user_id: int, name: str):
        self.conn.execute("INSERT OR IGNORE INTO users (user_id, name) VALUES (?, ?)", (user_id, name))
        self.conn.commit()

    def add_item(self, name: str, origin_era: str, max_duration: int):
        self.conn.execute("INSERT OR IGNORE INTO items (name, origin_era, max_duration) VALUES (?, ?, ?)",
                          (name, origin_era, max_duration))
        self.conn.commit()

    def borrow_item(self, user_id: int, item_name: str):
        cursor = self.conn.cursor()
        cursor.execute("SELECT item_id, is_borrowed FROM items WHERE name = ?", (item_name,))
        result = cursor.fetchone()
        if not result:
            print("Item not found.")
            return
        item_id, is_borrowed = result
        if is_borrowed:
            print("Item already borrowed.")
            return

        cursor.execute("UPDATE items SET is_borrowed = 1 WHERE item_id = ?", (item_id,))
        cursor.execute("INSERT OR IGNORE INTO borrowed_items (user_id, item_id) VALUES (?, ?)", (user_id, item_id))
        self.conn.commit()
        print(f"✅ Borrowed: {item_name}")

    def return_item(self, user_id: int, item_name: str):
        cursor = self.conn.cursor()
        cursor.execute("SELECT item_id FROM items WHERE name = ?", (item_name,))
        result = cursor.fetchone()
        if not result:
            print("Item not found.")
            return
        item_id = result[0]
        cursor.execute("DELETE FROM borrowed_items WHERE user_id = ? AND item_id = ?", (user_id, item_id))
        cursor.execute("UPDATE items SET is_borrowed = 0 WHERE item_id = ?", (item_id,))
        self.conn.commit()
        print(f"🔁 Returned: {item_name}")

    def list_items(self):
        cursor = self.conn.cursor()
        cursor.execute("SELECT name, origin_era, max_duration, is_borrowed FROM items")
        return cursor.fetchall()

    def advance_turn(self):
        cursor = self.conn.cursor()
        cursor.execute("UPDATE borrowed_items SET turns_borrowed = turns_borrowed + 1")
        self.conn.commit()

        cursor.execute('''
            SELECT u.name, i.name, i.max_duration, b.turns_borrowed
            FROM borrowed_items b
            JOIN users u ON b.user_id = u.user_id
            JOIN items i ON b.item_id = i.item_id
        ''')
        results = cursor.fetchall()
        for user_name, item_name, max_dur, turns in results:
            if turns > max_dur:
                print(f"⚠ TIME DISRUPTION: {user_name} held '{item_name}' too long!")

    def close(self):
        self.conn.close()

# -------------------------
# GAME MANAGER (Bridge)
# -------------------------
class GameManager:
    def __init__(self, db: DatabaseManager):
        self.db = db

    def add_user(self, user_id: int, name: str):
        self.db.add_user(user_id, name)

    def add_item(self, name: str, era: str, max_duration: int):
        self.db.add_item(name, era, max_duration)

    def borrow_item(self, user_id: int, item_name: str):
        self.db.borrow_item(user_id, item_name)

    def return_item(self, user_id: int, item_name: str):
        self.db.return_item(user_id, item_name)

    def next_turn(self):
        self.db.advance_turn()

# -------------------------
# MAIN CLI
# -------------------------
def main():
    db = DatabaseManager()
    gm = GameManager(db)

    # Setup sample data
    gm.add_user(1, "Anna the Time Traveler")
    gm.add_item("Laser Sword", "Future", 3)
    gm.add_item("Pharaoh's Amulet", "Ancient Egypt", 2)
    gm.add_item("Steam Gun", "Industrial Era", 4)

    while True:
        print("\n--- Time Traveler's Vault ---")
        print("1. View Items")
        print("2. Borrow Item")
        print("3. Return Item")
        print("4. Advance Time")
        print("5. Exit")
        print("6. Ask the Librarian (AI)")

        choice = input("Choose an option: ")

        if choice == "1":
            for name, era, dur, borrowed in db.list_items():
                status = "Borrowed" if borrowed else "Available"
                print(f"{name} ({era}) - Duration: {dur} - {status}")
        elif choice == "2":
            item_name = input("Enter item name to borrow: ")
            gm.borrow_item(1, item_name)
        elif choice == "3":
            item_name = input("Enter item name to return: ")
            gm.return_item(1, item_name)
        elif choice == "4":
            gm.next_turn()
        elif choice == "5":
            db.close()
            print("Goodbye, Time Traveler.")
            break
        elif choice == "6":
            question = input("Ask the AI Librarian: ")
            answer = ask_chatbot(question)
            print("\n📚 Librarian says:", answer)
        else:
            print("Invalid input.")

# Entry point
if __name__ == "__main__":
    main()




--- Time Traveler's Vault ---
1. View Items
2. Borrow Item
3. Return Item
4. Advance Time
5. Exit
6. Ask the Librarian (AI)


NameError: name 'GameManager' is not defined